In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/kal-klasifikasi-jenis-cuaca/sample_submission.csv
/kaggle/input/kal-klasifikasi-jenis-cuaca/train.csv
/kaggle/input/kal-klasifikasi-jenis-cuaca/test.csv


In [2]:
import os
print(os.listdir("/kaggle/input/"))

['kal-klasifikasi-jenis-cuaca']


In [3]:
print(os.listdir("/kaggle/input/kal-klasifikasi-jenis-cuaca"))

['sample_submission.csv', 'train.csv', 'test.csv']


In [4]:
import pandas as pd

data = pd.read_csv("/kaggle/input/kal-klasifikasi-jenis-cuaca/train.csv")
print(data.head())

   ID  Temperature  Humidity  Wind Speed  Precipitation (%)    Cloud Cover  \
0   0           14        73         9.5                 82  partly cloudy   
1   1           39        96         8.5                 71  partly cloudy   
2   2           30        64         7.0                 16          clear   
3   3           38        83         1.5                 82          clear   
4   4           27        74        17.0                 66       overcast   

   Atmospheric Pressure  UV Index  Season  Visibility (km)  Location  \
0               1010.82         2  Winter              3.5    inland   
1               1011.43         7  Spring             10.0    inland   
2               1018.72         5  Spring              5.5  mountain   
3               1026.25         7  Spring              1.0   coastal   
4                990.67         1  Winter              2.5  mountain   

  Weather Type  
0        Rainy  
1       Cloudy  
2        Sunny  
3        Sunny  
4        Rain

In [5]:
from sklearn.model_selection import train_test_split
data_latih, data_uji = train_test_split(data, test_size = 0.3, random_state = 101)
data_latih = data_latih.reset_index(drop = True)
data_uji = data_uji.reset_index(drop =True)

data.shape[0]

9240

In [6]:
print(data_uji.shape[0])
print(data_latih.shape[0])

2772
6468


In [7]:
from math import log2

def hitung_entropy(kolom_kelas):
  elemen, banyak = np.unique(kolom_kelas, return_counts=True)
  entropy = - (np.sum([(banyak[i] / np.sum(banyak)) * log2(banyak[i] / np.sum(banyak)) for i in range(len(elemen))]))
  return entropy

In [8]:
def hitung_ingormation_gain(data, nama_fitur_split, nama_fitur_kelas):
    nilai, banyak = np.unique(data[nama_fitur_split], return_counts=True)
    info_gain = hitung_entropy(data[nama_fitur_kelas])

    for i in range(len(nilai)):
        subset_entropy = hitung_entropy(data.where(data[nama_fitur_split] == nilai[i]).dropna()[nama_fitur_kelas])
        info_gain -= (banyak[i] / np.sum(banyak)) * subset_entropy

    return info_gain

In [9]:
from math import log2

def hitung_split_info(subset_sizes, total_samples):
  split_info = 0
  for size in subset_sizes:
    if size > 0 and total_samples > 0 and (size / total_samples) > 0:
      split_info -= (size / total_samples) * log2(size / total_samples)
  return split_info

In [10]:
def hitung_gain_ratio(data, nama_fitur_split, nama_fitur_kelas):
  hitung_ingormation_gain_value = hitung_ingormation_gain(data, nama_fitur_split, nama_fitur_kelas)
  subset_sizes = [len(data.where(data[nama_fitur_split] == nilai).dropna()) for nilai in np.unique(data[nama_fitur_split])]
  total_samples = len(data)
  hitung_split_info_value = hitung_split_info(subset_sizes, total_samples)

  if hitung_split_info_value == 0:
    return 0

  gain_ratio = hitung_ingormation_gain_value / hitung_split_info_value
  return gain_ratio

In [11]:
def buat_tree(data, data_awal, daftar_fitur, nama_fitur_kelas, kelas_parent_node = None):
  kelas = data[nama_fitur_kelas]
  if len(kelas.unique()) <= 1:
    return kelas.unique()[0]
  elif len(data) == 0:
    return kelas_parent_node
  elif len(daftar_fitur) == 0:
    return kelas.value_counts().sort_values(ascending = False).index[0]
  kelas_parent_node = kelas.value_counts().sort_values(ascending = False).index[0]
  gain_ratio = [hitung_gain_ratio(data, fitur, nama_fitur_kelas) for fitur in daftar_fitur]
  indeks_fitur_terbaik = np.argmax(gain_ratio)
  fitur_terbaik = daftar_fitur[indeks_fitur_terbaik]
  tree = {fitur_terbaik:{}}
  daftar_fitur = [i for i in daftar_fitur if i != fitur_terbaik]
  for nilai in data[fitur_terbaik].unique():
    sub_data = data.where(data[fitur_terbaik] == nilai).dropna()
    subtree = buat_tree(sub_data, data_awal, daftar_fitur, nama_fitur_kelas, kelas_parent_node)
    tree[fitur_terbaik][nilai] = subtree
  return tree

In [12]:
print(data.columns)

Index(['ID', 'Temperature', 'Humidity', 'Wind Speed', 'Precipitation (%)',
       'Cloud Cover', 'Atmospheric Pressure', 'UV Index', 'Season',
       'Visibility (km)', 'Location', 'Weather Type'],
      dtype='object')


In [13]:
tree = buat_tree(data_latih, data_latih, data_latih.columns[:-1], 'Humidity')

In [14]:
from pprint import pprint
pprint(tree)

{'Humidity': {20: 20.0,
              21: 21.0,
              22: 22.0,
              23: 23.0,
              24: 24.0,
              25: 25.0,
              26: 26.0,
              27: 27.0,
              28: 28.0,
              29: 29.0,
              30: 30.0,
              31: 31.0,
              32: 32.0,
              33: 33.0,
              34: 34.0,
              35: 35.0,
              36: 36.0,
              37: 37.0,
              38: 38.0,
              39: 39.0,
              40: 40.0,
              41: 41.0,
              42: 42.0,
              43: 43.0,
              44: 44.0,
              45: 45.0,
              46: 46.0,
              47: 47.0,
              48: 48.0,
              49: 49.0,
              50: 50.0,
              51: 51.0,
              52: 52.0,
              53: 53.0,
              54: 54.0,
              55: 55.0,
              56: 56.0,
              57: 57.0,
              58: 58.0,
              59: 59.0,
              60: 60.0,
              61

In [15]:
def prediksi(data_uji, tree):
  for key in list(data_uji.keys()):
    if key in list(tree.keys()):
      try:
        hasil = tree[key][data_uji[key]]
      except:
        return 1
      hasil = tree[key][data_uji[key]]
      if isinstance(hasil, dict):
        return prediksi (data_uji, hasil)
      else:
        return hasil

In [16]:
data_uji_dict = data_uji.iloc[:, :-1].to_dict(orient="records")

In [17]:
print(len(data_uji_dict))  # Pastikan list tidak kosong
print(data_uji_dict[:3])  # Cek 3 elemen pertama dari list

2772
[{'ID': 4637, 'Temperature': 32, 'Humidity': 51, 'Wind Speed': 0.0, 'Precipitation (%)': 28, 'Cloud Cover': 'partly cloudy', 'Atmospheric Pressure': 1000.21, 'UV Index': 1, 'Season': 'Winter', 'Visibility (km)': 5.5, 'Location': 'inland'}, {'ID': 356, 'Temperature': 23, 'Humidity': 28, 'Wind Speed': 5.0, 'Precipitation (%)': 0, 'Cloud Cover': 'partly cloudy', 'Atmospheric Pressure': 1016.71, 'UV Index': 11, 'Season': 'Winter', 'Visibility (km)': 10.0, 'Location': 'mountain'}, {'ID': 8615, 'Temperature': 22, 'Humidity': 67, 'Wind Speed': 4.5, 'Precipitation (%)': 37, 'Cloud Cover': 'overcast', 'Atmospheric Pressure': 1013.23, 'UV Index': 2, 'Season': 'Spring', 'Visibility (km)': 8.0, 'Location': 'inland'}]


In [18]:
hasil_prediksi_total = []
for i in range(len(data_uji_dict)):
  hasil_prediksi = prediksi(data_uji_dict[i], tree)
  hasil_prediksi_total.append(hasil_prediksi)

In [19]:
y_pred = hasil_prediksi_total
y_true = data_uji['Humidity']
benar = 0
for i in range(len(y_pred)):
  if y_pred[i] == y_true[i]:
    benar += 1
    print(f"Prediksi: {y_pred[i]}, Aktual: {y_true[i]}, Hasil: Benar")
  else:
    print(f"Prediksi: {y_pred[i]}, Aktual: {y_true[i]}, Hasil: Salah")
    benar -= 1
print(f"Jumlah prediksi benar: {benar}")
print(f"Jumlah prediksi salah: {len(y_pred) - benar}")
print(f"Akurasi: {benar / len(y_pred) * 100}%")

Prediksi: 51.0, Aktual: 51, Hasil: Benar
Prediksi: 28.0, Aktual: 28, Hasil: Benar
Prediksi: 67.0, Aktual: 67, Hasil: Benar
Prediksi: 77.0, Aktual: 77, Hasil: Benar
Prediksi: 65.0, Aktual: 65, Hasil: Benar
Prediksi: 92.0, Aktual: 92, Hasil: Benar
Prediksi: 53.0, Aktual: 53, Hasil: Benar
Prediksi: 65.0, Aktual: 65, Hasil: Benar
Prediksi: 75.0, Aktual: 75, Hasil: Benar
Prediksi: 71.0, Aktual: 71, Hasil: Benar
Prediksi: 30.0, Aktual: 30, Hasil: Benar
Prediksi: 70.0, Aktual: 70, Hasil: Benar
Prediksi: 33.0, Aktual: 33, Hasil: Benar
Prediksi: 63.0, Aktual: 63, Hasil: Benar
Prediksi: 74.0, Aktual: 74, Hasil: Benar
Prediksi: 21.0, Aktual: 21, Hasil: Benar
Prediksi: 91.0, Aktual: 91, Hasil: Benar
Prediksi: 67.0, Aktual: 67, Hasil: Benar
Prediksi: 49.0, Aktual: 49, Hasil: Benar
Prediksi: 25.0, Aktual: 25, Hasil: Benar
Prediksi: 96.0, Aktual: 96, Hasil: Benar
Prediksi: 77.0, Aktual: 77, Hasil: Benar
Prediksi: 61.0, Aktual: 61, Hasil: Benar
Prediksi: 72.0, Aktual: 72, Hasil: Benar
Prediksi: 72.0, 

In [20]:
print(type(data_uji_dict))

<class 'list'>


In [21]:
print(data_uji_dict[0])  # Lihat isi elemen pertama

{'ID': 4637, 'Temperature': 32, 'Humidity': 51, 'Wind Speed': 0.0, 'Precipitation (%)': 28, 'Cloud Cover': 'partly cloudy', 'Atmospheric Pressure': 1000.21, 'UV Index': 1, 'Season': 'Winter', 'Visibility (km)': 5.5, 'Location': 'inland'}


In [22]:
submission = pd.DataFrame({'Prediction': y_pred})
submission.to_csv("submission.csv", index=False, encoding="utf-8")